In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"  # 禁止联网，只用本地 cache

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# 本地数据集：repo_id 为子目录名，root 为父目录
dataset = LeRobotDataset(
    repo_id='/vla/.data/test',
    video_backend='torchcodec',
    )
dataset

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LeRobotDataset({
    Repository ID: '/vla/.data/test',
    Number of selected episodes: '1',
    Number of selected samples: '436',
    Features: '['observation.state', 'action', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

In [ ]:
import torch
from lerobot.utils.import_utils import register_third_party_plugins
from lerobot.policies.factory import make_policy, make_pre_post_processors
from lerobot_policy_my_policy import MyPolicyConfig

register_third_party_plugins()  # 必须：注册 lerobot_policy_my_policy 插件

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 方式 A：从零初始化（还没有训练好的 checkpoint）
config = MyPolicyConfig(device=str(device))
policy = make_policy(config, ds_meta=dataset.meta).eval()
policy

/mnt/workspace/luyi/.cache/miniconda3/envs/myvla/lib/python3.10/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


MyPolicyPolicy(
  (model): Sequential(
    (0): Linear(in_features=9, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=12, bias=True)
  )
)

In [ ]:
preprocess, postprocess = make_pre_post_processors(
    policy.config,
    dataset_stats=dataset.meta.stats,
    preprocessor_overrides={"device_processor": {"device": str(device)}},
)

print("policy type:", policy.config.type)
print("input features:", list(policy.config.input_features.keys()))

policy type: my_policy
input features: ['observation.state', 'observation.images.robot0_agentview_left_image', 'observation.images.robot0_agentview_right_image', 'observation.images.robot0_eye_in_hand_image']


### 推理验证

In [ ]:
batch = preprocess(dataset[0])
def to_device(batch):
    return {
        k: v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v
        for k, v in batch.items()
    }
batch = to_device(batch)

In [ ]:
with torch.inference_mode():
    pred_action_raw = policy.select_action(batch)
pred_action_raw

tensor([[ 0.0325, -0.1573,  0.0859,  0.2819, -0.1597, -0.0929, -0.0406,  0.3290,
         -0.1648,  0.0410,  0.0567,  0.4584]], device='cuda:0')

### 前向传播

In [ ]:
with torch.inference_mode(): # 不进行反向传播
    loss, output_dict = policy.forward(batch)
print("loss:", loss.item() if hasattr(loss, "item") else loss)
print("output keys:", output_dict.keys() if isinstance(output_dict, dict) else type(output_dict))

loss: 0.8428356647491455
output keys: dict_keys(['mse'])
